# Backpropagation

## Prepare your dataset

In [ ]:
import torch
import torchvision  #this library is for computer vision related things
import torchvision.transforms as transforms
import tqdm

def load_mnist_data(root_path='./data', batch_size=4):
    transform = transforms.Compose(  # transforms help to prepare the image tensor into appropriate format. compose helps to pack it all
        [transforms.ToTensor(),      #we are converting the image data into tensor form
        transforms.Normalize((0.5), (0.5))] # we are normalizing the tensor data
    )

    trainset = torchvision.datasets.MNIST(root=root_path, train=True, download=True, transform=transform) #we are loading image in root path . if not there we download.
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2) #based on batch size train loader loads the training batch from trainset. 

    testset = torchvision.datasets.MNIST(root=root_path, train=False, download=True, transform=transform)
    testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

    return trainloader, testloader

## Building your neural network

In [ ]:
import numpy as np
from typing import Any, Callable, Tuple

##################################
# For matrices or arbitrary size #
##################################
class MyWeightTensor:
    def __init__(self, shape: Tuple or int, init_weight_fn: Callable = np.random.randn, init_weights: 'MyWeightTensor' or np.ndarray or int or float = None):
        assert isinstance(shape, tuple) or isinstance(shape, int) or isinstance(shape, float), f'Allowed shapes: tuple, int, float, got: {type(shape)}'
        self.shape = shape

        if init_weights is not None: #if initial weights are passed then we initialize weight tensor with it. 
            if isinstance(init_weights, MyWeightTensor): # if initial weight is tensor(matrix) then we initialize it with values of the matrix
                self.values = init_weights.values
            else:
                if isinstance(shape, tuple): # if tuple(vector) or scalar then we assign accordingly
                    assert isinstance(init_weights, np.ndarray)
                else:
                    assert isinstance(init_weights, int) or isinstance(init_weights, float)
                
                self.values = init_weights
        else:
            if isinstance(shape, int): #if initial weight matrix is not passed, we initialize it with random normal distribution. we have initializer function for it which is already passed as default value 
                self.shape = (self.shape,)
                self.values = init_weight_fn(shape)
            else:
                self.values = init_weight_fn(*shape) #we cant pass directly tuple so we unpack it to create weight tensor(in this case vector)
    
    @property
    def T(self) -> 'MyWeightTensor':  # function for transpose. it returns matrix/vector(tensor)
        _T = self.values.T
        return MyWeightTensor(shape=_T.shape, init_weights=_T)
    
    def __add__(self, other) -> 'MyWeightTensor':
        if isinstance(other, MyWeightTensor):
            other = other.values
        else:
            assert isinstance(other, np.ndarray) or isinstance(other, int) or isinstance(other, float)
        
        return MyWeightTensor(shape=self.values.shape, init_weights=self.values + other)

    def __mul__(self, other) -> 'MyWeightTensor':
        if isinstance(other, MyWeightTensor):
            other = other.values
        else:
            assert isinstance(other, np.ndarray) or isinstance(other, int) or isinstance(other, float)
        
        _dot = np.dot(self.values, other)

        return MyWeightTensor(shape=_dot.shape, init_weights=_dot)

For creating a linear layer

In [ ]:
class MyLinearLayer:
    def __init__(self, in_features: int, out_features: int, init_weight_fn: Callable = np.random.randn) -> None:
        self.in_features = in_features
        self.out_features = out_features

        self.weights = MyWeightTensor(shape=(out_features, in_features), init_weight_fn=init_weight_fn)
        self.bias = MyWeightTensor(shape=out_features, init_weight_fn=init_weight_fn)

        self.latest_input = None
        self.latest_output = None

    def __call__(self, tensor: np.ndarray or MyWeightTensor) -> MyWeightTensor: 
        self.latest_input = tensor #we axpect tensor(matrix7vector) as input

        bs = -1 #we assume batch size is one dimensional(scalar). means only one sample is there
        if len(tensor.shape) == 2: # if it is two dimensional(means multiple sample are there, then input vector is now matrix )
            # batch size included
            bs = tensor.shape[0] # first dimension is always the sample size
            _w = self.weights * tensor.T #we need to transpose the input matrix otherwise multiplication is not possible
        else:
            _w = self.weights * tensor #if the sample is only one then we multiply weight matrix(in pytorch is dimension output * input so no need to transpose) with input vector
        
        _bias = self.bias.values
        if bs != -1:                      #if batch size is multi dimensional
            _bias = np.tile(_bias, bs).reshape(bs, -1) #if there are many sample then adding with bias is not possible due to dimension mismatch. so we elongate bias vector into matrix where
                                                       #each row is duplication of itself. it will have dimension of sample_size * output_features
        
        self.latest_output = (_w + _bias.T).T # output is weighted sum plus bias. we transpose the bias to add with weighted sum and the result is again transposed to bring back to form sample * features

        return MyWeightTensor(shape=self.latest_output.shape, init_weights=self.latest_output)
    
    def derivative(self) -> float:
        assert self.latest_output is not None, 'Cannot calculate grad without a single forward pass.'
        # Return a linear activation derivation
        # your code
        linear_derivation=1.0
        
        return linear_derivation
    
    #creating an activation function class for relu
    
class Relu:
    def __init__(self) -> None:
        self.latest_input = None


    def __call__(self, tensor: Any) -> MyWeightTensor:
        if isinstance(tensor, MyWeightTensor):
            data=tensor.values #access the raw numpy array
        else:
            data=tensor 
        
        self.latest_input=data

        #output value
        output_values=np.maximum(0,data) #relu activation function
        return MyWeightTensor(shape=output_values.shape, init_weights=output_values)
     

    @staticmethod
    def derivative(x: np.ndarray) -> np.ndarray:
        return (x > 0).astype(float)

In [ ]:
import numpy as np
def xavier_normal_init(*shape) -> np.ndarray:          #we use xavier initialization to properly initialize the weight to avoid vanishing/exploding gradient 
    assert len(shape) <= 2, 'Can only init max 2d tensors'
    fan_in = shape[0] #input features
    if len(shape) == 1:
        fan_out = fan_in  #when it is 1-d then input and output dimension the same
    else:
        fan_out = shape[1] #output feature is the second dim
    gain = 1.0             #it is just the scaling factor. example for relu activation we use gain=sqrt(2)

    std = gain * np.sqrt(2.0 / (fan_in + fan_out)) #this is standard deviation . xavier ~ N(0, 2.0/(input_features + output_features))
    return np.random.normal(loc=0.0, scale=std, size=shape)


class MyNeuralNetwork:
    def __init__(self) -> None:
        # init_weight_fn = lambda *shape: np.random.randn(*shape) / 10
        init_weight_fn = lambda *shape: xavier_normal_init(*shape)
        
        # Build you own neural network with list of MyLinearLayer 
        # Note numbers of input and final output features of the neural network
        # your code
        #solution:
        # we have here three layers, first layer takes 784 input features and outputs 128 features, 
        # second layer takes 128 input features and outputs 64 features, third layer takes 64 input features 
        # and outputs 10 features (for 10 classes)

        self.layers = [MyLinearLayer(in_features=784, out_features=128, init_weight_fn=init_weight_fn),
                       Relu(),
                       MyLinearLayer(in_features=128, out_features=64, init_weight_fn=init_weight_fn),
                       Relu(),
                       MyLinearLayer(in_features=64, out_features=10, init_weight_fn=init_weight_fn)]
    
    def __call__(self, tensor: np.ndarray) -> Any:
        x = tensor
        for layer in self.layers:
            x = layer(x)
        
        return x

## Implement your loss function(s)

In [25]:
import torch.nn.functional as F

class CrossEntropyLoss:
    def __init__(self) -> None:
        pass

    def __call__(self, predictions: MyWeightTensor or np.ndarray, targets: MyWeightTensor or np.ndarray) -> np.ndarray:
        """
        Computes cross entropy between targets snd predictions.    
        Returns: List of cross entropy losses (batch-wise)
        """
        if isinstance(predictions, MyWeightTensor):
            predictions = predictions.values
        
        if isinstance(targets, MyWeightTensor):
            targets = targets.values

        assert predictions.shape[0] == targets.shape[0]
        if len(targets.shape) == 2:
            targets = targets.reshape(-1)
        predictions = torch.as_tensor(predictions)
        targets = torch.as_tensor(targets)

        loss = np.array([F.cross_entropy(pred, t).item() for pred, t in zip(predictions, targets)])

        return loss
    
    def derivative(self) -> Callable:
        # y_hat is the prediction
        # y is the target value
        def _derivative(y_hat: MyWeightTensor or np.ndarray, y: MyWeightTensor or np.ndarray) -> np.ndarray:
            if isinstance(y_hat, MyWeightTensor):
                y_hat = y_hat.values
            
            if isinstance(y, MyWeightTensor):
                y = y.values


            # implement the derivation of the cross entropy 
            # your code
            #solution: lets create a small value to prevent division by zero
            epsilon = 1e-12
            return -y / (y_hat+ epsilon)
        
        return _derivative   

## Implement the training loop

In [30]:
def train(model: MyNeuralNetwork, batch_size: int, learning_rate: float, loss_fn: Callable, epochs: int = 10):
    train_loader, _ = load_mnist_data(batch_size=batch_size)

    for epoch in range(epochs):
        running_loss = 0.0
        running_accuracy = []
        for imgs, targets in tqdm.tqdm(train_loader, desc=f'Training iteration {epoch + 1}'):

            # for custom model
            imgs = imgs.numpy()
            targets = targets.numpy()

            if len(targets.shape) == 1:
                targets = targets.reshape(-1, 1)

            imgs = imgs.reshape(-1, 28 * 28)

            imgs = MyWeightTensor(shape=imgs.shape, init_weights=imgs)

            outputs = model(imgs).values

            loss = loss_fn(outputs, targets)
            avg_loss = np.mean(loss)

            # print statistics
            running_loss += avg_loss

            # Calculate the Accuracy (how many of all samples are correctly classified?)
            max_outputs = np.argmax(outputs, axis=1)
            accuracy = (max_outputs == targets.flatten()).mean()
            running_accuracy.append(accuracy)

            #########################
            # Start backpropagation #
            #########################

            # Your code for backpropagation!
              
            
            # Step 1: starting from the very end with loss function
            # beginning with gradients of the loss
            grad_loss = loss_fn.derivative()(outputs, targets)
            # Step 2: start back propagating the neural network 
            for layer in reversed(model.layers):
                if isinstance(layer, MyLinearLayer):
                # for each layer calculate the output gradient and update weights and bias
                    grad_loss = grad_loss * layer.derivative()  # chain rule
                    # Update weights and bias using the calculated gradients and learning rate
                    d_weights = np.dot(grad_loss.T, layer.latest_input.values)
                    d_bias = np.sum(grad_loss, axis=0) 

                    new_grad_loss= np.dot(grad_loss, layer.weights.values)
                    # Update weights and bias
                    layer.weights.values -= learning_rate * d_weights
                    layer.bias.values -= learning_rate * d_bias
                    grad_loss = new_grad_loss  # propagate the gradient to the previous layer

                elif isinstance(layer, Relu):
                    # The derivative of ReLU is 1 if input > 0, else 0
                    mask = layer.derivative(layer.latest_input)
                    
                    # Element-wise multiplication (Gradient only flows where input was positive)
                    grad_loss = grad_loss * mask
            #######################
            # End backpropagation #
            #######################

        print(f'Epoch {epoch + 1} finished with loss: {running_loss / len(train_loader):.3f} and accuracy: {torch.tensor(running_accuracy).mean():.3f}')

In [ ]:
#############################
# Execute the training loop #
#############################
model =  MyNeuralNetwork()
batch_size = 4
learning_rate = 0.001
epochs = 10
loss_fn = CrossEntropyLoss()

train(
    model=model,
    batch_size=batch_size,
    learning_rate=learning_rate,
    epochs=epochs,
    loss_fn=loss_fn
)